# Kaggle S6E3 — Customer Churn v2
Added: ORIG dataset stats + DIGIT features + ROUND features (no slow TE interactions)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')
SEED = 42; N_SPLITS = 5
np.random.seed(SEED)
print('ok')

In [ ]:
import os
if os.path.exists('/kaggle/input'):
    TRAIN_PATH = '/kaggle/input/competitions/playground-series-s6e3/train.csv'
    TEST_PATH  = '/kaggle/input/competitions/playground-series-s6e3/test.csv'
    SUB_PATH   = '/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv'
    ORIG_PATH  = '/kaggle/input/datasets/cdeotte/s6e3-original-dataset/WA_Fn-UseC_-Telco-Customer-Churn.csv'
else:
    TRAIN_PATH = 'data/train.csv'
    TEST_PATH  = 'data/test.csv'
    SUB_PATH   = 'data/sample_submission.csv'
    ORIG_PATH  = 'data/WA_Fn-UseC_-Telco-Customer-Churn.csv'

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
orig  = pd.read_csv(ORIG_PATH)
sub   = pd.read_csv(SUB_PATH)
print('train', train.shape, '| test', test.shape, '| orig', orig.shape)

In [ ]:
TARGET='Churn'
CATS=['gender','SeniorCitizen','Partner','Dependents','PhoneService',
    'MultipleLines','InternetService','OnlineSecurity','OnlineBackup','DeviceProtection',
    'TechSupport','StreamingTV','StreamingMovies','Contract','PaperlessBilling','PaymentMethod']

train_ids=train['id'].copy(); test_ids=test['id'].copy()
train=train.drop(columns=['id']); test=test.drop(columns=['id'])
if 'customerID' in orig.columns: orig=orig.drop(columns=['customerID'])

def map_target(s): return s.map({'Yes':1,'No':0}).astype('int8') if s.dtype==object else s.astype('int8')
train[TARGET]=map_target(train[TARGET]); orig[TARGET]=map_target(orig[TARGET])

for df in [train,test,orig]:
    df['tenure']=pd.to_numeric(df['tenure'],errors='coerce').astype('float32')
    df['MonthlyCharges']=pd.to_numeric(df['MonthlyCharges'],errors='coerce').astype('float32')
    df['TotalCharges']=pd.to_numeric(df['TotalCharges'].astype(str).str.strip().replace('',np.nan),errors='coerce').astype('float32')
    m=df['TotalCharges'].isna()
    df.loc[m,'TotalCharges']=df.loc[m,'tenure']*df.loc[m,'MonthlyCharges']
    for c in CATS:
        if c in df.columns: df[c]=df[c].astype(str).fillna('Missing')

BASE=[c for c in train.columns if c!=TARGET]
print('BASE cols:',len(BASE),'| churn rate:',round(train[TARGET].mean(),4))

In [ ]:
from itertools import combinations

# ── DIGIT features ────────────────────────────────────────────
def add_digit_cols(df, col, multiplier):
    if multiplier == 'direct':
        vals = pd.to_numeric(df[col], errors='coerce').fillna(-1).astype(int)
    else:
        vals = (pd.to_numeric(df[col], errors='coerce') * multiplier).round(0).fillna(-1).astype(int)
    max_len = {'tenure':3,'MonthlyCharges':5,'TotalCharges':6}.get(col, vals.astype(str).str.len().max())
    s = vals.astype(str).str.zfill(max_len)
    created = []
    for i in range(max_len):
        n = f'{col}_DIGIT_{i+1}'; df[n] = s.str[i].astype(int); created.append(n)
    return created

DIGIT = []
for df in [train, test]:
    d = add_digit_cols(df, 'tenure', 'direct')
    add_digit_cols(df, 'MonthlyCharges', 100)
    add_digit_cols(df, 'TotalCharges', 1)
    if df is train:
        DIGIT = d + [f'MonthlyCharges_DIGIT_{i+1}' for i in range(5)] + [f'TotalCharges_DIGIT_{i+1}' for i in range(6)]
print(len(DIGIT), 'DIGIT features')

# ── ROUND features ────────────────────────────────────────────
ROUND = []
for col, levels in [('MonthlyCharges',[('1s',0),('10s',-1),('100s',-2),('1000s',-3)]),
                    ('TotalCharges',  [('1s',0),('10s',-1),('100s',-2),('1000s',-3)]),
                    ('tenure',        [('1s',0),('10s',-1)])]:
    for suffix, level in levels:
        n = f'{col}_ROUND_{suffix}'; ROUND.append(n)
        for df in [train, test]: df[n] = df[col].round(level).fillna(-1).astype(int)
print(len(ROUND), 'ROUND features')

# ── ORIG dataset stats ────────────────────────────────────────
ORIG_FEATS = []
orig_mean = float(orig[TARGET].mean())
for col in BASE:
    mm = orig.groupby(col)[TARGET].mean()
    cm = orig.groupby(col).size()
    for df in [train, test]:
        df[f'orig_mean_{col}']  = df[col].map(mm).fillna(orig_mean)
        df[f'orig_count_{col}'] = df[col].map(cm).fillna(0)
    ORIG_FEATS += [f'orig_mean_{col}', f'orig_count_{col}']
print(len(ORIG_FEATS), 'ORIG features')

# ── INTERACTION features (string concat, TE inside CV) ────────
INTER = []
for col1, col2 in combinations(BASE, 2):
    n = f'{col1}_{col2}'; INTER.append(n)
    for df in [train, test]: df[n] = df[col1].astype(str) + '_' + df[col2].astype(str)
print(len(INTER), 'INTER features')

FEATURES = list(dict.fromkeys(BASE + ORIG_FEATS + INTER + ROUND + DIGIT))
print(len(FEATURES), 'total features')

In [ ]:
from sklearn.model_selection import KFold
import time

def factorize3(tr, val, te):
    codes, _ = pd.factorize(pd.concat([tr, val, te], ignore_index=True).astype(str))
    n1, n2 = len(tr), len(val)
    return (pd.Series(codes[:n1], index=tr.index, dtype='int32'),
            pd.Series(codes[n1:n1+n2], index=val.index, dtype='int32'),
            pd.Series(codes[n1+n2:], index=te.index, dtype='int32'))

# ── Precompute TE on interactions ONCE (outside CV) ───────────
print('Precomputing target encoding for 171 interactions...')
t0 = time.time()
X_all = train[FEATURES].copy()
y_all = train[TARGET].copy()
Xtest_raw = test[FEATURES].copy()
gm = float(y_all.mean())

oof_te = pd.DataFrame(index=X_all.index)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for fold_i, (ti, vi) in enumerate(kf.split(X_all), 1):
    Xt = X_all.iloc[ti]; yt = y_all.iloc[ti]; Xv = X_all.iloc[vi]
    fm = float(yt.mean())
    for col in INTER:
        mp = Xt.assign(_y=yt.values).groupby(col)['_y'].mean()
        oof_te.loc[Xv.index, f'TE_{col}'] = Xv[col].map(mp).fillna(fm).values
    print(f'  TE fold {fold_i}/5 done — {time.time()-t0:.0f}s elapsed')

print('Computing test TE...')
full_map = {col: X_all.assign(_y=y_all.values).groupby(col)['_y'].mean() for col in INTER}
test_te = pd.DataFrame({f'TE_{col}': Xtest_raw[col].map(full_map[col]).fillna(gm) for col in INTER},
                       index=Xtest_raw.index)

X_all = X_all.drop(columns=INTER)
Xtest_raw = Xtest_raw.drop(columns=INTER)
for col in INTER:
    X_all[f'TE_{col}']     = oof_te[f'TE_{col}']
    Xtest_raw[f'TE_{col}'] = test_te[f'TE_{col}']

print(f'TE done in {time.time()-t0:.0f}s. Final feature count: {X_all.shape[1]}')

# ── CV ────────────────────────────────────────────────────────
lgb_params = {
    'objective':'binary','metric':'auc','n_estimators':2000,'learning_rate':0.05,
    'num_leaves':127,'subsample':0.8,'colsample_bytree':0.6,'min_child_samples':20,
    'reg_alpha':0.1,'reg_lambda':1.0,'random_state':SEED,'n_jobs':-1,'verbose':-1
}

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
oof = np.zeros(len(X_all), dtype=np.float32)
tpred = np.zeros(len(Xtest_raw), dtype=np.float32)
cv_start = time.time()

for fold, (tri, vali) in enumerate(skf.split(X_all, y_all), 1):
    fold_start = time.time()
    print(f'\n--- Fold {fold}/{N_SPLITS} ---')
    Xtr = X_all.iloc[tri].copy(); Xval = X_all.iloc[vali].copy()
    ytr = y_all.iloc[tri].copy(); yval = y_all.iloc[vali].copy()
    Xte = Xtest_raw.copy()

    print(f'  Factorizing categoricals...')
    for c in CATS:
        if c in Xtr.columns: Xtr[c], Xval[c], Xte[c] = factorize3(Xtr[c], Xval[c], Xte[c])
    for c in Xtr.select_dtypes('object').columns:
        Xtr[c], Xval[c], Xte[c] = factorize3(Xtr[c], Xval[c], Xte[c])

    print(f'  Training LightGBM on {Xtr.shape[0]:,} rows x {Xtr.shape[1]} features...')
    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(Xtr, ytr, eval_set=[(Xval, yval)],
              callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(False)])

    oof[vali] = model.predict_proba(Xval)[:, 1]
    tpred += model.predict_proba(Xte)[:, 1] / N_SPLITS

    fold_auc = roc_auc_score(yval, oof[vali])
    print(f'  Fold {fold} AUC: {fold_auc:.5f}  best_iter={model.best_iteration_}  time={time.time()-fold_start:.0f}s')

print(f'\nOOF AUC: {roc_auc_score(y_all, oof):.5f}  total_time={time.time()-cv_start:.0f}s')

In [ ]:
np.save('oof_lgb.npy', oof); np.save('pred_lgb.npy', tpred)
print(f'LightGBM OOF AUC: {roc_auc_score(y_all, oof):.5f}')

In [ ]:
from catboost import CatBoostClassifier

# CatBoost handles categoricals natively — pass cat_features indices
cat_cols = [c for c in CATS if c in X_all.columns]
cat_idx  = [X_all.columns.tolist().index(c) for c in cat_cols]

cb_params = dict(
    iterations=2000, learning_rate=0.05, depth=6,
    l2_leaf_reg=3, random_seed=SEED,
    eval_metric='AUC', od_type='Iter', od_wait=100,
    verbose=False, thread_count=-1,
)

oof_cat  = np.zeros(len(X_all), dtype=np.float32)
pred_cat = np.zeros(len(Xtest_raw), dtype=np.float32)
cv_start = time.time()

for fold, (tri, vali) in enumerate(skf.split(X_all, y_all), 1):
    fold_start = time.time()
    Xtr  = X_all.iloc[tri].copy();  Xval = X_all.iloc[vali].copy()
    ytr  = y_all.iloc[tri].copy();  yval = y_all.iloc[vali].copy()
    Xte  = Xtest_raw.copy()

    # CatBoost needs string cats — already string in X_all
    model_cb = CatBoostClassifier(**cb_params)
    model_cb.fit(Xtr, ytr, cat_features=cat_idx,
                 eval_set=(Xval, yval), use_best_model=True)

    oof_cat[vali] = model_cb.predict_proba(Xval)[:, 1]
    pred_cat += model_cb.predict_proba(Xte)[:, 1] / N_SPLITS

    print(f'CatBoost Fold {fold} AUC: {roc_auc_score(yval, oof_cat[vali]):.5f}  '
          f'best_iter={model_cb.best_iteration_}  time={time.time()-fold_start:.0f}s')

print(f'\nCatBoost OOF AUC: {roc_auc_score(y_all, oof_cat):.5f}  total={time.time()-cv_start:.0f}s')
np.save('oof_cat.npy', oof_cat); np.save('pred_cat.npy', pred_cat)

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

# Encode cats as integers for MLP, scale all features
def prep_for_nn(Xtr, Xval, Xte, cat_cols):
    Xtr = Xtr.copy(); Xval = Xval.copy(); Xte = Xte.copy()
    for c in cat_cols:
        codes, _ = pd.factorize(pd.concat([Xtr[c], Xval[c], Xte[c]], ignore_index=True).astype(str))
        n1, n2 = len(Xtr), len(Xval)
        Xtr[c]  = codes[:n1].astype('int32')
        Xval[c] = codes[n1:n1+n2].astype('int32')
        Xte[c]  = codes[n1+n2:].astype('int32')
    scaler = StandardScaler()
    Xtr  = pd.DataFrame(scaler.fit_transform(Xtr),  columns=Xtr.columns)
    Xval = pd.DataFrame(scaler.transform(Xval),     columns=Xval.columns)
    Xte  = pd.DataFrame(scaler.transform(Xte),      columns=Xte.columns)
    return Xtr, Xval, Xte

nn_params = dict(
    hidden_layer_sizes=(256, 128, 64),
    activation='relu', solver='adam',
    max_iter=200, early_stopping=True,
    validation_fraction=0.1, n_iter_no_change=15,
    random_state=SEED, verbose=False,
)

oof_nn  = np.zeros(len(X_all), dtype=np.float32)
pred_nn = np.zeros(len(Xtest_raw), dtype=np.float32)
cv_start = time.time()

for fold, (tri, vali) in enumerate(skf.split(X_all, y_all), 1):
    fold_start = time.time()
    Xtr  = X_all.iloc[tri].copy();  Xval = X_all.iloc[vali].copy()
    ytr  = y_all.iloc[tri].copy();  yval = y_all.iloc[vali].copy()
    Xte  = Xtest_raw.copy()

    Xtr, Xval, Xte = prep_for_nn(Xtr, Xval, Xte, cat_cols)

    model_nn = MLPClassifier(**nn_params)
    model_nn.fit(Xtr, ytr)

    oof_nn[vali] = model_nn.predict_proba(Xval)[:, 1]
    pred_nn += model_nn.predict_proba(Xte)[:, 1] / N_SPLITS

    print(f'NN Fold {fold} AUC: {roc_auc_score(yval, oof_nn[vali]):.5f}  '
          f'iters={model_nn.n_iter_}  time={time.time()-fold_start:.0f}s')

print(f'\nNN OOF AUC: {roc_auc_score(y_all, oof_nn):.5f}  total={time.time()-cv_start:.0f}s')
np.save('oof_nn.npy', oof_nn); np.save('pred_nn.npy', pred_nn)

In [ ]:
# ── Hill Climbing Ensemble ────────────────────────────────────
models = {
    'lgb': (oof,       tpred),
    'cat': (oof_cat,   pred_cat),
    'nn':  (oof_nn,    pred_nn),
}

print('Single model OOF AUCs:')
for name, (o, _) in models.items():
    print(f'  {name}: {roc_auc_score(y_all, o):.5f}')

best_name = max(models, key=lambda n: roc_auc_score(y_all, models[n][0]))
best_auc  = roc_auc_score(y_all, models[best_name][0])
print(f'\nStarting from: {best_name} (AUC={best_auc:.5f})')

weights = {best_name: 1.0}
blend_oof  = models[best_name][0].copy().astype(float)
blend_pred = models[best_name][1].copy().astype(float)

TOL = 1e-5
for step in range(20):
    best_step_auc = best_auc
    best_w = None; best_add = None
    for name, (o, _) in models.items():
        for w in np.arange(0.05, 1.0, 0.05):
            trial = (blend_oof + w * o) / (1 + w)
            auc = roc_auc_score(y_all, trial)
            if auc > best_step_auc:
                best_step_auc = auc; best_w = w; best_add = name
        # try negative weights too
        for w in np.arange(0.05, 0.5, 0.05):
            trial = (blend_oof - w * o) / (1 - w)
            trial = np.clip(trial, 0, 1)
            auc = roc_auc_score(y_all, trial)
            if auc > best_step_auc:
                best_step_auc = auc; best_w = -w; best_add = name

    if best_add is None or (best_step_auc - best_auc) < TOL:
        print(f'Stopped: improvement {best_step_auc - best_auc:.8f} < TOL={TOL}')
        break

    w = best_w
    blend_oof  = (blend_oof  + w * models[best_add][0]) / (1 + w)
    blend_pred = (blend_pred + w * models[best_add][1]) / (1 + w)
    blend_oof  = np.clip(blend_oof, 0, 1)
    blend_pred = np.clip(blend_pred, 0, 1)
    weights[best_add] = weights.get(best_add, 0) + w
    best_auc = best_step_auc
    print(f'  Step {step}: AUC {best_auc:.6f} | add {best_add} w={w:.3f}')

print(f'\nFinal ensemble OOF AUC: {roc_auc_score(y_all, blend_oof):.5f}')
print('Weights:', weights)

sub['Churn'] = blend_pred
sub.to_csv('submission_ensemble.csv', index=False)
print('Saved submission_ensemble.csv')